In [21]:
# VLM Evaluation
import os

import os

def get_folder_names_os(path):
    """
    Returns a list of folder names within the specified path using the os module.
    """
    folder_names = []
    for entry in os.listdir(path):
        full_path = os.path.join(path, entry)
        if os.path.isdir(full_path):
            folder_names.append(entry)
    return folder_names

# Example usage:
target_directory = "../results/drone_sim/"  # Replace with your actual folder path
folders = get_folder_names_os(target_directory)

# folders.remove('.archive')
# folders.remove('detections')

In [31]:
import json

results_data = []
vlm_responses = {}
llm_responses = {}

for folder in folders:
    results_file = f'{target_directory}{folder}/results.json'
    vlm_responses.update({folder[11:-9]: []})
    llm_responses.update({folder[11:-9]: []})

    try:
        with open(results_file, 'r') as file:
            data = json.load(file)
        
        for result in data:
            # results_data.append({'folder': folder, 'data': result})
            vlm_responses[folder[11:-9]].append(result['vlm_description'])
            llm_responses[folder[11:-9]].append(result['assistance_instructions'])


        # print(results_data)
    except FileNotFoundError:
        print(f'{results_file} not found. Confirm file path or results file exsists.')
    except json.JSONDecodeError:
        print(f'{results_file} could not be decoded. Check file format.')
    except Exception as e:
        print(f'Unexcepted error {e}')

# for resp in vlm_responses:
#     for vlm_desc in vlm_responses[resp]:
#         print(vlm_desc)


In [32]:
evaluation_references = 'evaluation_references.json'

try:
    with open(evaluation_references, 'r') as file:
        references = json.load(file)
    # print(results_data)
except FileNotFoundError:
    print(f'{evaluation_references} not found. Confirm file path or results file exsists.')

In [20]:
import sacrebleu

In [28]:
bleu_scores = {}

for ref in references:
    world_and_scenario = ref['world_and_scenario']
    vlm_description_ref = ref['vlm_description']
    llm_description_ref = ref['llm_response']
    
    # print(, )
    vlm_bleu = sacrebleu.corpus_bleu(vlm_responses[world_and_scenario], vlm_description_ref)
    llm_bleu = sacrebleu.corpus_bleu(llm_responses[world_and_scenario], llm_description_ref)
    print(f'VLM BLEU Score: {vlm_bleu} \nLLM BLEU Score: {llm_bleu}')

VLM BLEU Score: BLEU = 0.81 2.2/1.1/0.6/0.3 (BP = 1.000 ratio = 45.000 hyp_len = 45 ref_len = 1) 
LLM BLEU Score: BLEU = 0.16 0.8/0.2/0.1/0.0 (BP = 1.000 ratio = 264.000 hyp_len = 264 ref_len = 1)
VLM BLEU Score: BLEU = 1.12 5.1/1.3/0.7/0.3 (BP = 1.000 ratio = 39.000 hyp_len = 39 ref_len = 1) 
LLM BLEU Score: BLEU = 0.29 1.4/0.3/0.2/0.1 (BP = 1.000 ratio = 148.000 hyp_len = 148 ref_len = 1)
VLM BLEU Score: BLEU = 0.90 4.2/1.1/0.5/0.3 (BP = 1.000 ratio = 48.000 hyp_len = 48 ref_len = 1) 
LLM BLEU Score: BLEU = 0.26 1.2/0.3/0.2/0.1 (BP = 1.000 ratio = 163.000 hyp_len = 163 ref_len = 1)
VLM BLEU Score: BLEU = 0.95 4.3/1.1/0.6/0.3 (BP = 1.000 ratio = 46.000 hyp_len = 46 ref_len = 1) 
LLM BLEU Score: BLEU = 0.25 1.2/0.3/0.1/0.1 (BP = 1.000 ratio = 171.000 hyp_len = 171 ref_len = 1)
VLM BLEU Score: BLEU = 0.90 4.2/1.1/0.5/0.3 (BP = 1.000 ratio = 48.000 hyp_len = 48 ref_len = 1) 
LLM BLEU Score: BLEU = 0.22 1.4/0.2/0.1/0.1 (BP = 1.000 ratio = 210.000 hyp_len = 210 ref_len = 1)
VLM BLEU Score:

In [34]:
from bert_score import score

# bert_score dict format:
# {
#   'world_and_scenario' : ''
#   , 'References' : {'VLM': '', LLM: ''}
#   , 'Candidates' : {'VLM': [], LLM: []}
#   , 'Bert_Scores' : {'VLM' : {'P': [], 'R': [], 'F1': []}, 'LLM' : {'P': [], 'R': [], 'F1': []}}
# }
bert_scores = []

# {
#   'world_and_scenario': ''
#   'LLM_Reference' : ''
#   'LLM_Candidate' : ''
#   'LLM_Precision' : ''
#   'LLM_Recall' : ''
#   'LLM_F1 Score' : ''
#   'Person_Found_match': ('True Positive', 'True Negative', 'False Positive', 'False Negative')
#   'Assistance_Required_Match' : ('True Positive', 'True Negative', 'False Positive', 'False Negative')
# }
LLM_bert_scores = []

# {
#   'world_and_scenario': ''
#   'VLM_Reference' : ''
#   'VLM_Candidate' : ''
#   'VLM_Precision' : ''
#   'VLM_Recall' : ''
#   'VLM_F1 Score' : ''
# }
VLM_bert_scores = []


for ref in references:
    world_and_scenario = ref['world_and_scenario']
    vlm_ref = ref['vlm_description']
    llm_ref = ref['llm_response']
    vlm_P, vlm_R, vlm_F1 = score(vlm_responses[world_and_scenario], [vlm_ref] * len(vlm_responses[world_and_scenario]), lang='en')
    llm_P, llm_R, llm_F1 = score(llm_responses[world_and_scenario], [llm_ref] * len(llm_responses[world_and_scenario]), lang='en')
    vlm_Precision_scores = []
    vlm_Recall_scores = []
    vlm_F1_scores = []

    llm_Precision_scores = []
    llm_Recall_scores = []
    llm_F1_scores = []

    for i, cand in enumerate(vlm_responses[world_and_scenario]):
        vlm_Precision_scores.append(vlm_P[i].item())
        vlm_Recall_scores.append(vlm_R[i].item())
        vlm_F1_scores.append(vlm_F1[i].item())

        VLM_bert_scores.append({'world_scenario' : world_and_scenario
                                , 'VLM_Reference' : vlm_ref
                                , 'VLM_Candidate' : cand
                                , 'VLM_Precision' : vlm_P[i].item()
                                , 'VLM_Recall': vlm_R[i].item()
                                , 'VLM_F1_Score' : vlm_F1[i].item()
                                })

    for i, cand in enumerate(llm_responses[world_and_scenario]):
        llm_Precision_scores.append(llm_P[i].item())
        llm_Recall_scores.append(llm_R[i].item())
        llm_F1_scores.append(llm_F1[i].item())

        if '1. Person Found' in llm_ref and '1. Person Found' in cand:
            person_found_match = 'True Positive'
        elif '1. Person Found' in llm_ref and '1. No Person Found' in cand:
            person_found_match = 'False Negative'
        elif '1. No Person Found' in llm_ref and '1. No Person Found' in cand:
            person_found_match = 'True Negative'
        elif '1. No Person Found' in llm_ref and '1. Person Found' in cand:
            person_found_match = 'False Positive'

        if '2. Person Requires Immediate Assistance' in llm_ref and '2. Person Requires Immediate Assistance' in cand:
            assistance_required_match = 'True Positive'
        elif '2. Person Requires Immediate Assistance' in llm_ref and '2. Person does not require assistance' in cand:
            assistance_required_match = 'False Negative'
        elif '2. Person does not require assistance' in llm_ref and '2. Person does not require assistance' in cand:
            assistance_required_match = 'True Negative'
        elif '2. Person does not require assistance' in llm_ref and '2. Person Requires Immediate Assistance' in cand:
            assistance_required_match = 'False Positive'

        LLM_bert_scores.append({'world_scenario' : world_and_scenario
                                , 'LLM_Reference' : llm_ref
                                , 'LLM_Candidate' : cand
                                , 'LLM_Precision' : llm_P[i].item()
                                , 'LLM_Recall': llm_R[i].item()
                                , 'LLM_F1_Score' : llm_F1[i].item()
                                , 'Person_Found_Match': person_found_match
                                , 'Assistance_Required_Match': assistance_required_match
                                })

    eval_bert_results = {'world_and_scenario': world_and_scenario
                         , 'References': {'VLM': vlm_ref, 'LLM': llm_ref}
                         , 'Candidates': {'VLM': vlm_responses[world_and_scenario], 'LLM': llm_responses[world_and_scenario]}
                         , 'Bert_Scores': {
                                        'VLM': {'Precision': vlm_Precision_scores, 'Recall': vlm_Recall_scores, 'F1': vlm_F1_scores}
                                        , 'LLM': {'Precision': llm_Precision_scores, 'Recall': llm_Recall_scores, 'F1': llm_F1_scores}
                                }
                         }
    
    bert_scores.append(eval_bert_results)


/Users/davidlelis/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initializ

In [59]:
print(bert_scores)

[{'world_and_scenario': 'world1_scenario1', 'References': {'VLM': 'Person standing in a parking lot', 'LLM': 'Person Found. No Assistance Needed. Person misdialed emergency services and is not injured or in danger'}, 'Candidates': {'VLM': ['The image is a  of a square with a white line in the center. The square is rectangular in shape and appears to be made of a smooth, glossy material. The background is a solid black color. On the left side', 'The image is a  of a person standing in front of a black background with a white square in the center. The person is wearing a blue suit and has their arms stretched out to the side, as if they are reaching for', 'The image shows a  of a person standing in front of a black background. The person is wearing a blue suit and has short dark hair. They are standing with their arms stretched out to the sides, as if they are reaching', 'The image shows a  of a young man standing in front of a black background. He is wearing a blue suit and has short da

In [35]:
vlm_precision_scores = []
vlm_recall_scores = []
vlm_F1_scores = []
for score in bert_scores:
    for i in score['Bert_Scores']['VLM']['Precision']:
        vlm_precision_scores.append(i)
    for i in score['Bert_Scores']['VLM']['Recall']:
        vlm_recall_scores.append(i)
    for i in score['Bert_Scores']['VLM']['F1']:
        vlm_F1_scores.append(i)

avg_vlm_precision_scores = sum(vlm_precision_scores) / len(vlm_precision_scores)
avg_vlm_recall_scores = sum(vlm_recall_scores) / len(vlm_recall_scores)
avg_vlm_F1_scores = sum(vlm_F1_scores) / len(vlm_F1_scores)

print(avg_vlm_precision_scores, avg_vlm_recall_scores, avg_vlm_F1_scores)

0.8240661045338246 0.8931990894865482 0.8570522473213521


In [14]:
for score in VLM_bert_scores:
    print(score)

{'world_scenario': 'world1_scenario1', 'VLM_Reference': 'Person standing in a parking lot', 'VLM_Candidate': 'The image is a  of a square with a white line in the center. The square is rectangular in shape and appears to be made of a smooth, glossy material. The background is a solid black color. On the left side', 'VLM_Precision': 0.7966315150260925, 'VLM_Recall': 0.8173573017120361, 'VLM_F1_Score': 0.8068612813949585}
{'world_scenario': 'world1_scenario1', 'VLM_Reference': 'Person standing in a parking lot', 'VLM_Candidate': 'The image is a  of a person standing in front of a black background with a white square in the center. The person is wearing a blue suit and has their arms stretched out to the side, as if they are reaching for', 'VLM_Precision': 0.7973476052284241, 'VLM_Recall': 0.8423093557357788, 'VLM_F1_Score': 0.8192119598388672}
{'world_scenario': 'world1_scenario1', 'VLM_Reference': 'Person standing in a parking lot', 'VLM_Candidate': 'The image shows a  of a person stand

In [26]:
llm_precision_scores = []
llm_recall_scores = []
llm_F1_scores = []
for score in bert_scores:
    for i in score['Bert_Scores']['LLM']['Precision']:
        llm_precision_scores.append(i)
    for i in score['Bert_Scores']['LLM']['Recall']:
        llm_recall_scores.append(i)
    for i in score['Bert_Scores']['LLM']['F1']:
        llm_F1_scores.append(i)

avg_llm_precision_scores = sum(llm_precision_scores) / len(llm_precision_scores)
avg_llm_recall_scores = sum(llm_recall_scores) / len(llm_recall_scores)
avg_llm_F1_scores = sum(llm_F1_scores) / len(llm_F1_scores)

print(avg_llm_precision_scores, avg_llm_recall_scores, avg_llm_F1_scores)

0.8348552772339354 0.8904009542566664 0.8615679799242223


In [36]:
import pandas as pd

In [37]:
llm_bert_df = pd.DataFrame(LLM_bert_scores)
vlm_bert_df = pd.DataFrame(VLM_bert_scores)

In [38]:
llm_bert_df.to_excel('LLM_Bert_Scores.xlsx', index=False)
vlm_bert_df.to_excel('VLM_Bert_Scores.xlsx', index=False)